# Dataset profiling

In [3]:
# ============================================================
# DATASET PROFILING (ONE CELL) — Recod.ai/LUC Scientific Image Forgery
# Fix: handle duplicate case_id in train images (hw_lookup no longer requires unique index)
#
# DINO_DIR:
#   /kaggle/input/dinov2/pytorch/base/1
#
# Output:
# - /kaggle/working/recodai_luc_prof/image_profile.parquet
# - /kaggle/working/recodai_luc_prof/mask_index.parquet
# - /kaggle/working/recodai_luc_prof/mask_profile.parquet
# - /kaggle/working/recodai_luc_prof/paths.json
# ============================================================

import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ----------------------------
# 0) Fixed DINO path
# ----------------------------
DINO_DIR = Path("/kaggle/input/dinov2/pytorch/base/1")
need_files = ["config.json", "pytorch_model.bin", "preprocessor_config.json"]
dino_ok = DINO_DIR.exists() and all((DINO_DIR / f).exists() for f in need_files)
print("DINO_DIR:", str(DINO_DIR), "| OK:", bool(dino_ok))
if not dino_ok:
    missing = [f for f in need_files if not (DINO_DIR / f).exists()]
    print("  Missing:", missing)

# ----------------------------
# 1) Auto-detect COMP_ROOT
# ----------------------------
def find_comp_root():
    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError("Not on Kaggle: /kaggle/input not found.")
    cands = []
    for d in base.iterdir():
        if not d.is_dir():
            continue
        ss = d / "sample_submission.csv"
        if ss.exists() and (d / "train_images").exists() and (d / "test_images").exists():
            cands.append(d)
    if not cands:
        for d in base.iterdir():
            if not d.is_dir():
                continue
            for ss in d.rglob("sample_submission.csv"):
                root = ss.parent
                if (root / "train_images").exists() and (root / "test_images").exists():
                    cands.append(root)
                    break
    if not cands:
        raise FileNotFoundError("Cannot find competition root under /kaggle/input")
    return sorted(cands, key=lambda x: len(str(x)))[0]

COMP_ROOT = find_comp_root()
TRAIN_IMG_AUTH = COMP_ROOT / "train_images" / "authentic"
TRAIN_IMG_FORG = COMP_ROOT / "train_images" / "forged"
TRAIN_MASK_DIR = COMP_ROOT / "train_masks"
SUP_IMG_DIR = COMP_ROOT / "supplemental_images"
SUP_MASK_DIR = COMP_ROOT / "supplemental_masks"
TEST_IMG_DIR = COMP_ROOT / "test_images"
SAMPLE_SUB = COMP_ROOT / "sample_submission.csv"

OUT_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "COMP_ROOT": str(COMP_ROOT),
    "SAMPLE_SUB": str(SAMPLE_SUB),
    "TRAIN_IMG_AUTH": str(TRAIN_IMG_AUTH),
    "TRAIN_IMG_FORG": str(TRAIN_IMG_FORG),
    "TRAIN_MASK_DIR": str(TRAIN_MASK_DIR),
    "SUP_IMG_DIR": str(SUP_IMG_DIR),
    "SUP_MASK_DIR": str(SUP_MASK_DIR),
    "TEST_IMG_DIR": str(TEST_IMG_DIR),
    "DINO_DIR": str(DINO_DIR),
    "OUT_DIR": str(OUT_DIR),
}
(OUT_DIR / "paths.json").write_text(json.dumps(PATHS, indent=2))

print("COMP_ROOT:", COMP_ROOT)
print("OUT_DIR  :", OUT_DIR)
print("-"*60)

# ----------------------------
# Helpers
# ----------------------------
def stem_case_id(p: Path):
    m = re.match(r"^(\d+)", p.stem)
    return int(m.group(1)) if m else p.stem

def fast_image_meta(p: Path):
    try:
        with Image.open(p) as im:
            w, h = im.size
            mode = im.mode
        return h, w, mode, None
    except Exception as e:
        return None, None, None, str(e)[:200]

def bbox_from_bool(mask_bool: np.ndarray):
    ys, xs = np.where(mask_bool)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    return (x1, y1, x2, y2)

# ----------------------------
# 2) Image profiling
# ----------------------------
img_rows = []

for p in sorted(TRAIN_IMG_AUTH.glob("*.png")):
    img_rows.append({"split":"train","source":"train","label":"authentic","case_id":stem_case_id(p),"img_path":str(p)})
for p in sorted(TRAIN_IMG_FORG.glob("*.png")):
    img_rows.append({"split":"train","source":"train","label":"forged","case_id":stem_case_id(p),"img_path":str(p)})

if SUP_IMG_DIR.exists():
    for p in sorted(SUP_IMG_DIR.glob("*.png")):
        img_rows.append({"split":"train","source":"supplemental","label":"unknown","case_id":stem_case_id(p),"img_path":str(p)})

for p in sorted(TEST_IMG_DIR.glob("*.png")):
    img_rows.append({"split":"test","source":"test","label":"unknown","case_id":stem_case_id(p),"img_path":str(p)})

df_img = pd.DataFrame(img_rows).drop_duplicates(subset=["split","img_path"]).reset_index(drop=True)

metas = []
for p in df_img["img_path"].map(Path):
    h,w,mode,err = fast_image_meta(p)
    metas.append((h,w,mode,err))
df_img[["H","W","mode","img_err"]] = pd.DataFrame(metas, columns=["H","W","mode","img_err"])
df_img["aspect"] = df_img["W"] / df_img["H"]
df_img["is_gray"] = df_img["mode"].isin(["L","LA"])

# report duplicate case_id in train split (allowed; we just handle it)
train_counts = df_img.query("split=='train'").groupby("case_id").size()
n_dup_cases = int((train_counts > 1).sum())
if n_dup_cases > 0:
    print("WARNING: duplicate case_id in train split:", n_dup_cases, "cases (handled)")

# build hw_lookup safely (no unique-index requirement)
df_train_hw = df_img.query("split=='train'")[["case_id","H","W","img_path"]].dropna(subset=["H","W"]).copy()
hw_lookup = (
    df_train_hw.sort_values("img_path")
               .groupby("case_id")[["H","W"]]
               .first()
               .to_dict("index")
)

df_img.to_parquet(OUT_DIR / "image_profile.parquet", index=False)

print("IMAGES:")
print(df_img.groupby(["split","source","label"]).size().rename("n").reset_index())
print("Gray ratio (train):", float(df_img.query("split=='train'")["is_gray"].mean()))
print("-"*60)

# ----------------------------
# 3) Mask profiling (instance-aware)
# ----------------------------
mask_files = []
if TRAIN_MASK_DIR.exists():
    mask_files += list(TRAIN_MASK_DIR.glob("*.npy"))
if SUP_MASK_DIR.exists():
    mask_files += list(SUP_MASK_DIR.glob("*.npy"))
mask_files = sorted(mask_files)

mask_rows = []

for mp in mask_files:
    case_id = stem_case_id(mp)
    src = "train" if mp.parent.name == "train_masks" else "supplemental"
    try:
        arr = np.load(mp, mmap_mode="r")
    except Exception as e:
        mask_rows.append({
            "case_id": case_id, "source": src, "mask_path": str(mp),
            "inst_id": -1, "K": None, "H": None, "W": None,
            "area_px": None, "area_frac": None,
            "bbox_x1": None, "bbox_y1": None, "bbox_x2": None, "bbox_y2": None,
            "shape_mismatch": None, "mask_err": str(e)[:200]
        })
        continue

    if arr.ndim == 2:
        insts = [arr]
    elif arr.ndim == 3:
        insts = [arr[i] for i in range(arr.shape[0])]
    else:
        mask_rows.append({
            "case_id": case_id, "source": src, "mask_path": str(mp),
            "inst_id": -1, "K": None, "H": None, "W": None,
            "area_px": None, "area_frac": None,
            "bbox_x1": None, "bbox_y1": None, "bbox_x2": None, "bbox_y2": None,
            "shape_mismatch": None, "mask_err": f"Unexpected ndim={arr.ndim}"
        })
        continue

    K = len(insts)
    for i, m in enumerate(insts):
        m_bool = (np.asarray(m) > 0)
        Hm, Wm = m_bool.shape
        area = int(np.count_nonzero(m_bool))
        bbox = bbox_from_bool(m_bool)
        if bbox is None:
            x1=y1=x2=y2=None
        else:
            x1,y1,x2,y2 = bbox

        sm = None
        if case_id in hw_lookup:
            Hi, Wi = int(hw_lookup[case_id]["H"]), int(hw_lookup[case_id]["W"])
            sm = int((Hi != Hm) or (Wi != Wm))

        denom = float(Hm*Wm) if (Hm and Wm) else None
        area_frac = (area/denom) if denom else None

        mask_rows.append({
            "case_id": case_id, "source": src, "mask_path": str(mp),
            "inst_id": i, "K": K, "H": Hm, "W": Wm,
            "area_px": area, "area_frac": area_frac,
            "bbox_x1": x1, "bbox_y1": y1, "bbox_x2": x2, "bbox_y2": y2,
            "shape_mismatch": sm, "mask_err": None
        })

df_mask_idx = pd.DataFrame(mask_rows)
df_mask_idx.to_parquet(OUT_DIR / "mask_index.parquet", index=False)

valid = df_mask_idx[df_mask_idx["mask_err"].isna()].copy()
agg = valid.groupby(["case_id","source"]).agg(
    n_instances=("inst_id","count"),
    union_area_px=("area_px","sum"),   # overcounts if overlap; OK for profiling
    max_inst_area_px=("area_px","max"),
    mean_inst_area_px=("area_px","mean"),
    mean_area_frac=("area_frac","mean"),
    max_area_frac=("area_frac","max"),
    any_shape_mismatch=("shape_mismatch", lambda x: int(np.nanmax(x.fillna(0))) if len(x) else 0),
    any_empty_inst=("area_px", lambda x: int((x.fillna(0)==0).any()))
).reset_index()

agg.to_parquet(OUT_DIR / "mask_profile.parquet", index=False)

print("MASKS:")
print("mask files:", len(mask_files))
print("cases with masks:", int(agg["case_id"].nunique()) if len(agg) else 0)
print("instances (rows):", int(len(valid)))
print("shape mismatches (cases):", int(agg["any_shape_mismatch"].sum()) if len(agg) else 0)
print("cases with empty instance:", int(agg["any_empty_inst"].sum()) if len(agg) else 0)
print("-"*60)

# quantiles snapshot
if len(agg):
    q = agg["max_area_frac"].dropna()
    if len(q):
        print("max_area_frac quantiles:", q.quantile([0,0.25,0.5,0.75,0.9,0.95,0.99]).to_dict())

# update supplemental labels: if supplemental has mask => forged
if SUP_IMG_DIR.exists() and len(agg):
    sup_cases_with_mask = set(agg.loc[agg["source"].eq("supplemental"), "case_id"].tolist())
    m = (df_img["source"].eq("supplemental")) & (df_img["split"].eq("train"))
    df_img.loc[m, "label"] = df_img.loc[m, "case_id"].map(lambda x: "forged" if x in sup_cases_with_mask else "unknown")
    df_img.to_parquet(OUT_DIR / "image_profile.parquet", index=False)

print("DONE. Saved:")
print(" -", OUT_DIR / "paths.json")
print(" -", OUT_DIR / "image_profile.parquet")
print(" -", OUT_DIR / "mask_index.parquet")
print(" -", OUT_DIR / "mask_profile.parquet")


DINO_DIR: /kaggle/input/dinov2/pytorch/base/1 | OK: True
COMP_ROOT: /kaggle/input/recodai-luc-scientific-image-forgery-detection
OUT_DIR  : /kaggle/working/recodai_luc_prof
------------------------------------------------------------
IMAGES:
   split        source      label     n
0   test          test    unknown     1
1  train  supplemental    unknown    48
2  train         train  authentic  2377
3  train         train     forged  2751
Gray ratio (train): 0.06027820710973725
------------------------------------------------------------
MASKS:
mask files: 2799
cases with masks: 2795
instances (rows): 3909
shape mismatches (cases): 4
cases with empty instance: 0
------------------------------------------------------------
max_area_frac quantiles: {0.0: 0.0003236437680472542, 0.25: 0.005805188221578222, 0.5: 0.02361882716049383, 0.75: 0.07490765765765767, 0.9: 0.12696696696696697, 0.95: 0.17178184361133758, 0.99: 0.3362424620059605}
DONE. Saved:
 - /kaggle/working/recodai_luc_prof/paths.

# Data, Labels, CV & Sanity Guards

In [4]:
# ============================================================
# STAGE — Data, Labels, CV & Sanity Guards (ONE CELL) — RECOD.ai/LUC
# Reads outputs from Dataset Profiling (recodai_luc_prof) and builds:
# - df_train_all (unique case_id, label y, chosen img_path, mask stats)
# - df_test (unique case_id, chosen img_path, aligned to sample_submission)
# - folds (StratifiedKFold by y)
#
# Output:
# - /kaggle/working/recodai_luc_prof/train_manifest.parquet
# - /kaggle/working/recodai_luc_prof/test_manifest.parquet
# - /kaggle/working/recodai_luc_prof/folds.parquet
# - /kaggle/working/recodai_luc_prof/sanity_report.json
# - /kaggle/working/recodai_luc_prof/dup_case_images.csv
# ============================================================

import os, json, re
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Config
# ----------------------------
N_FOLDS = 5
SEED = 42
STRICT = False  # set True to raise on critical issues

OUT_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths_json = OUT_DIR / "paths.json"
if not paths_json.exists():
    raise FileNotFoundError(f"Missing {paths_json}. Run Dataset Profiling cell first.")

PATHS = json.loads(paths_json.read_text())
COMP_ROOT = Path(PATHS["COMP_ROOT"])
SAMPLE_SUB = Path(PATHS["SAMPLE_SUB"])

# Inputs from profiling
img_prof_pq  = OUT_DIR / "image_profile.parquet"
mask_idx_pq  = OUT_DIR / "mask_index.parquet"
mask_prof_pq = OUT_DIR / "mask_profile.parquet"
for p in [img_prof_pq, mask_idx_pq, mask_prof_pq]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run Dataset Profiling cell first.")

df_img = pd.read_parquet(img_prof_pq)
df_mask_idx = pd.read_parquet(mask_idx_pq)
df_mask_prof = pd.read_parquet(mask_prof_pq)

# sample_submission defines test ids order
df_sub = pd.read_csv(SAMPLE_SUB)
if "case_id" not in df_sub.columns:
    raise ValueError("sample_submission.csv must contain 'case_id' column")
sub_case_ids = df_sub["case_id"].tolist()

# ----------------------------
# Helpers
# ----------------------------
def _prefer_path(rows: pd.DataFrame, want: str):
    # want: "forged" or "authentic" or "any"
    if want == "forged":
        cand = rows[rows["img_path"].astype(str).str.contains(r"/forged/|\\forged\\", regex=True)]
        if len(cand): return cand.sort_values("img_path").iloc[0]
    if want == "authentic":
        cand = rows[rows["img_path"].astype(str).str.contains(r"/authentic/|\\authentic\\", regex=True)]
        if len(cand): return cand.sort_values("img_path").iloc[0]
    return rows.sort_values("img_path").iloc[0]

def resolve_train_case(group: pd.DataFrame, has_mask: bool):
    # label resolution priority: mask -> forged; else use folder label vote
    labels = group["label"].astype(str).tolist()
    any_forged_folder = any(l == "forged" for l in labels)
    any_auth_folder = any(l == "authentic" for l in labels)

    if has_mask or any_forged_folder:
        y = 1
        label = "forged"
        pick = _prefer_path(group, "forged")
    else:
        y = 0
        label = "authentic"
        pick = _prefer_path(group, "authentic") if any_auth_folder else _prefer_path(group, "any")

    out = {
        "case_id": int(group["case_id"].iloc[0]),
        "y": int(y),
        "label": label,
        "img_path": str(pick["img_path"]),
        "source": str(pick.get("source", "train")),
        "H": pick.get("H", None),
        "W": pick.get("W", None),
        "mode": pick.get("mode", None),
        "is_gray": bool(pick.get("is_gray", False)) if pd.notna(pick.get("is_gray", False)) else False,
        "aspect": float(pick.get("aspect", np.nan)) if pd.notna(pick.get("aspect", np.nan)) else np.nan,
        "n_img_paths": int(len(group)),
        "dup_img_paths": int(len(group) > 1),
        "img_err_any": int(group["img_err"].notna().any()) if "img_err" in group.columns else 0,
    }
    # keep list for debugging
    out["_all_img_paths"] = "|".join(sorted(group["img_path"].astype(str).tolist()))
    out["_all_labels"] = "|".join(sorted(set(labels)))
    return out

def resolve_test_case(group: pd.DataFrame):
    pick = _prefer_path(group, "any")
    out = {
        "case_id": int(group["case_id"].iloc[0]),
        "img_path": str(pick["img_path"]),
        "H": pick.get("H", None),
        "W": pick.get("W", None),
        "mode": pick.get("mode", None),
        "is_gray": bool(pick.get("is_gray", False)) if pd.notna(pick.get("is_gray", False)) else False,
        "aspect": float(pick.get("aspect", np.nan)) if pd.notna(pick.get("aspect", np.nan)) else np.nan,
        "n_img_paths": int(len(group)),
        "dup_img_paths": int(len(group) > 1),
        "img_err_any": int(group["img_err"].notna().any()) if "img_err" in group.columns else 0,
        "_all_img_paths": "|".join(sorted(group["img_path"].astype(str).tolist())),
    }
    return out

# mask availability per case_id (train+supp)
mask_cases = set(df_mask_prof["case_id"].astype(int).unique().tolist())
mask_prof_map = df_mask_prof.set_index(["case_id","source"]).copy()

# ----------------------------
# 1) Build df_train_all (unique case_id)
# ----------------------------
df_train_img = df_img[df_img["split"].eq("train")].copy()
if df_train_img.empty:
    raise RuntimeError("No train images found in image_profile.parquet")

train_rows = []
dup_debug_rows = []

for cid, g in df_train_img.groupby("case_id", sort=True):
    cid_int = int(cid)
    has_mask = cid_int in mask_cases
    rec = resolve_train_case(g, has_mask=has_mask)
    train_rows.append(rec)
    if rec["dup_img_paths"] == 1:
        dup_debug_rows.append({"case_id": cid_int, "labels_seen": rec["_all_labels"], "paths_seen": rec["_all_img_paths"]})

df_train_all = pd.DataFrame(train_rows)

# Join mask stats (prefer train source then supplemental if exists)
def pick_mask_stats(cid: int):
    # try (cid,"train") then (cid,"supplemental") then none
    if (cid, "train") in mask_prof_map.index:
        r = mask_prof_map.loc[(cid, "train")]
        return r
    if (cid, "supplemental") in mask_prof_map.index:
        r = mask_prof_map.loc[(cid, "supplemental")]
        return r
    return None

mask_stats = []
for cid in df_train_all["case_id"].astype(int).tolist():
    r = pick_mask_stats(cid)
    if r is None:
        mask_stats.append((0, 0, np.nan, np.nan, 0, 0))
    else:
        mask_stats.append((
            int(r.get("n_instances", 0)),
            int(r.get("union_area_px", 0)),
            float(r.get("max_area_frac", np.nan)),
            float(r.get("mean_area_frac", np.nan)),
            int(r.get("any_shape_mismatch", 0)),
            int(r.get("any_empty_inst", 0)),
        ))

df_train_all[["n_instances","union_area_px","max_area_frac","mean_area_frac","any_shape_mismatch","any_empty_inst"]] = \
    pd.DataFrame(mask_stats, columns=["n_instances","union_area_px","max_area_frac","mean_area_frac","any_shape_mismatch","any_empty_inst"])

df_train_all["has_mask"] = (df_train_all["n_instances"] > 0).astype(int)

# Guard: forged must have mask (in train)
bad_forg_no_mask = df_train_all.query("y==1 and has_mask==0")
# Guard: mask shape mismatch
bad_shape = df_train_all.query("any_shape_mismatch==1")

# ----------------------------
# 2) Build df_test aligned to sample_submission
# ----------------------------
df_test_img = df_img[df_img["split"].eq("test")].copy()
if df_test_img.empty:
    raise RuntimeError("No test images found in image_profile.parquet")

test_rows = []
for cid, g in df_test_img.groupby("case_id", sort=True):
    test_rows.append(resolve_test_case(g))
df_test = pd.DataFrame(test_rows)

# align to sample_submission order (keep all)
df_test = df_test.set_index("case_id")
missing_in_test = [cid for cid in sub_case_ids if cid not in df_test.index]
extra_in_test = [cid for cid in df_test.index.tolist() if cid not in set(sub_case_ids)]

if len(missing_in_test) > 0:
    msg = f"Missing {len(missing_in_test)} test case_id(s) found in sample_submission but not in test_images (first 10: {missing_in_test[:10]})"
    if STRICT: raise RuntimeError(msg)
    print("WARNING:", msg)

# Create aligned df_test_ordered with NaNs for missing (won't crash later stages if handled)
df_test_ordered = df_test.reindex(sub_case_ids).reset_index()

# ----------------------------
# 3) CV folds (StratifiedKFold on y)
# ----------------------------
try:
    from sklearn.model_selection import StratifiedKFold
except Exception as e:
    raise RuntimeError("sklearn not available; cannot build CV folds") from e

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = np.full(len(df_train_all), -1, dtype=np.int32)
y = df_train_all["y"].astype(int).values

for f, (_, va_idx) in enumerate(skf.split(np.zeros_like(y), y)):
    folds[va_idx] = f

df_train_all["fold"] = folds.astype(int)

# ----------------------------
# 4) Save artifacts + sanity report
# ----------------------------
if dup_debug_rows:
    pd.DataFrame(dup_debug_rows).to_csv(OUT_DIR / "dup_case_images.csv", index=False)
else:
    (OUT_DIR / "dup_case_images.csv").write_text("case_id,labels_seen,paths_seen\n")

df_train_all.drop(columns=["_all_img_paths","_all_labels"], errors="ignore").to_parquet(OUT_DIR / "train_manifest.parquet", index=False)
df_test_ordered.drop(columns=["_all_img_paths"], errors="ignore").to_parquet(OUT_DIR / "test_manifest.parquet", index=False)

df_folds = df_train_all[["case_id","fold","y"]].copy()
df_folds.to_parquet(OUT_DIR / "folds.parquet", index=False)

report = {
    "n_train_cases": int(df_train_all["case_id"].nunique()),
    "n_test_cases_profiled": int(df_test.shape[0]),
    "n_test_cases_sample_submission": int(len(sub_case_ids)),
    "n_test_missing_vs_sample_submission": int(len(missing_in_test)),
    "n_test_extra_vs_sample_submission": int(len(extra_in_test)),
    "train_y_mean": float(df_train_all["y"].mean()),
    "train_dup_case_images": int((df_train_all["dup_img_paths"]==1).sum()),
    "train_img_err_any": int((df_train_all["img_err_any"]==1).sum()),
    "forged_no_mask_cases": int(len(bad_forg_no_mask)),
    "shape_mismatch_cases": int(len(bad_shape)),
    "empty_instance_cases": int((df_train_all["any_empty_inst"]==1).sum()),
    "fold_counts": df_train_all["fold"].value_counts().sort_index().to_dict(),
    "fold_pos_counts": df_train_all.groupby("fold")["y"].sum().astype(int).to_dict(),
}
(OUT_DIR / "sanity_report.json").write_text(json.dumps(report, indent=2))

print("SAVED:")
print(" -", OUT_DIR / "train_manifest.parquet")
print(" -", OUT_DIR / "test_manifest.parquet")
print(" -", OUT_DIR / "folds.parquet")
print(" -", OUT_DIR / "sanity_report.json")
print(" -", OUT_DIR / "dup_case_images.csv")
print("-"*60)
print("SANITY:", json.dumps({k: report[k] for k in [
    "n_train_cases","train_y_mean","train_dup_case_images","forged_no_mask_cases","shape_mismatch_cases",
    "n_test_missing_vs_sample_submission"
]}, indent=2))

if len(bad_forg_no_mask) > 0:
    msg = f"Found forged-labeled cases without masks: {len(bad_forg_no_mask)} (see train_manifest.parquet)"
    if STRICT: raise RuntimeError(msg)
    print("WARNING:", msg)

if len(bad_shape) > 0:
    msg = f"Found mask/image shape mismatches: {len(bad_shape)} (see train_manifest.parquet)"
    if STRICT: raise RuntimeError(msg)
    print("WARNING:", msg)


SAVED:
 - /kaggle/working/recodai_luc_prof/train_manifest.parquet
 - /kaggle/working/recodai_luc_prof/test_manifest.parquet
 - /kaggle/working/recodai_luc_prof/folds.parquet
 - /kaggle/working/recodai_luc_prof/sanity_report.json
 - /kaggle/working/recodai_luc_prof/dup_case_images.csv
------------------------------------------------------------
SANITY: {
  "n_train_cases": 2795,
  "train_y_mean": 1.0,
  "train_dup_case_images": 2377,
  "forged_no_mask_cases": 0,
  "shape_mismatch_cases": 4,
  "n_test_missing_vs_sample_submission": 0
}


# DINOv2 Feature Cache 

In [5]:
# ============================================================
# STAGE — DINOv2 Feature Cache (CPU-Optimized) (ONE CELL)
# - Fixed model path: /kaggle/input/dinov2/pytorch/base/1
# - Fixed input size: 518x518  -> token grid 37x37 (patch=14)
#
# Requires (from previous stages):
# - /kaggle/working/recodai_luc_prof/paths.json
# - /kaggle/working/recodai_luc_prof/train_manifest.parquet
# - /kaggle/working/recodai_luc_prof/test_manifest.parquet
#
# Output:
# - /kaggle/working/recodai_luc/cache/dinov2_base_518_cfg_<hash>/{train,test}/{case_id}.npz
# - /kaggle/working/recodai_luc/cache/dinov2_base_518_cfg_<hash>/tokens_manifest_{train,test}.parquet
# - /kaggle/working/recodai_luc/cache/dinov2_base_518_cfg_<hash>/cfg.json
# ============================================================

import os, gc, json, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch

# ----------------------------
# Config
# ----------------------------
DINO_DIR = Path("/kaggle/input/dinov2/pytorch/base/1")
IMG_SIZE = 518            # fixed square
PATCH = 14                # DINOv2 ViT patch size
HTOK = IMG_SIZE // PATCH  # 37
WTOK = IMG_SIZE // PATCH  # 37
SAVE_DTYPE = "float16"    # reduce disk usage
BATCH = 8                 # CPU batch
NUM_WORKERS = 0           # keep 0 (simple + stable)

assert IMG_SIZE % PATCH == 0, "IMG_SIZE must be divisible by PATCH"

# ----------------------------
# Load manifests
# ----------------------------
PROF_DIR = Path("/kaggle/working/recodai_luc_prof")
paths_json = PROF_DIR / "paths.json"
train_pq = PROF_DIR / "train_manifest.parquet"
test_pq  = PROF_DIR / "test_manifest.parquet"
for p in [paths_json, train_pq, test_pq]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run previous stages first.")

PATHS = json.loads(paths_json.read_text())

df_train = pd.read_parquet(train_pq)
df_test  = pd.read_parquet(test_pq)

# ----------------------------
# Transformers load (local)
# ----------------------------
try:
    from transformers import AutoModel, AutoImageProcessor
    processor = AutoImageProcessor.from_pretrained(str(DINO_DIR), local_files_only=True)
except Exception:
    from transformers import AutoModel, AutoFeatureExtractor
    processor = AutoFeatureExtractor.from_pretrained(str(DINO_DIR), local_files_only=True)

model = AutoModel.from_pretrained(str(DINO_DIR), local_files_only=True)
model.eval()

device = torch.device("cpu")
model.to(device)

# normalization params
mean = np.array(getattr(processor, "image_mean", [0.485, 0.456, 0.406]), dtype=np.float32)
std  = np.array(getattr(processor, "image_std",  [0.229, 0.224, 0.225]), dtype=np.float32)

# ----------------------------
# Cache dirs (cfg-hashed)
# ----------------------------
CFG = {
    "dino_dir": str(DINO_DIR),
    "img_size": IMG_SIZE,
    "patch": PATCH,
    "htok": HTOK,
    "wtok": WTOK,
    "save_dtype": SAVE_DTYPE,
    "normalize_mean": mean.tolist(),
    "normalize_std": std.tolist(),
    "batch": BATCH,
}
cfg_id = hashlib.sha1(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:12]

CACHE_ROOT = Path("/kaggle/working/recodai_luc/cache") / f"dinov2_base_518_cfg_{cfg_id}"
TRAIN_OUT = CACHE_ROOT / "train"
TEST_OUT  = CACHE_ROOT / "test"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)
(CACHE_ROOT / "cfg.json").write_text(json.dumps(CFG, indent=2))

print("CACHE_ROOT:", CACHE_ROOT)
print("Train cases:", len(df_train), "| Test cases:", len(df_test))

# ----------------------------
# Helpers
# ----------------------------
def load_and_preprocess(p: str) -> np.ndarray:
    # return CHW float32 normalized
    with Image.open(p) as im:
        im = im.convert("RGB")
        im = im.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)
        x = np.asarray(im, dtype=np.float32) / 255.0  # HWC
    x = (x - mean) / std
    x = np.transpose(x, (2, 0, 1))  # CHW
    return x

@torch.no_grad()
def encode_batch(x_bchw: torch.Tensor) -> torch.Tensor:
    # returns patch tokens (B, HTOK, WTOK, D)
    out = model(pixel_values=x_bchw)
    h = out.last_hidden_state  # (B, 1+N, D)
    patch = h[:, 1:, :]        # remove CLS
    Bn, N, D = patch.shape
    if N != HTOK * WTOK:
        raise RuntimeError(f"Token count mismatch: got N={N}, expected {HTOK*WTOK} (check IMG_SIZE/PATCH)")
    patch = patch.reshape(Bn, HTOK, WTOK, D)
    return patch

def save_npz(dst: Path, tok_hw_d: np.ndarray):
    dst.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(dst, tok=tok_hw_d)

def run_cache(df: pd.DataFrame, out_dir: Path, split_name: str) -> pd.DataFrame:
    rows = []
    t0 = time.time()
    miss = 0
    done = 0
    skip = 0

    case_ids = df["case_id"].astype(int).tolist()
    img_paths = df["img_path"].astype(str).tolist()

    # process in fixed-size batches
    buf_case, buf_path, buf_x = [], [], []

    def flush():
        nonlocal done, skip
        if not buf_x:
            return
        x = torch.from_numpy(np.stack(buf_x, axis=0)).to(device)  # (B,3,H,W) float32
        tok = encode_batch(x).cpu().numpy()
        if SAVE_DTYPE == "float16":
            tok = tok.astype(np.float16)
        elif SAVE_DTYPE == "float32":
            tok = tok.astype(np.float32)
        else:
            raise ValueError("SAVE_DTYPE must be float16 or float32")

        for i in range(tok.shape[0]):
            cid = int(buf_case[i])
            pth = buf_path[i]
            dst = out_dir / f"{cid}.npz"
            if dst.exists():
                skip += 1
            else:
                save_npz(dst, tok[i])
                done += 1
            rows.append({
                "case_id": cid,
                "split": split_name,
                "img_path": pth,
                "npz_path": str(dst),
                "htok": HTOK,
                "wtok": WTOK,
                "dtype": SAVE_DTYPE,
            })

        buf_case.clear(); buf_path.clear(); buf_x.clear()
        gc.collect()

    for idx, (cid, pth) in enumerate(zip(case_ids, img_paths), start=1):
        p = Path(pth)
        if not p.exists():
            miss += 1
            rows.append({"case_id": int(cid), "split": split_name, "img_path": pth, "npz_path": None,
                         "htok": HTOK, "wtok": WTOK, "dtype": SAVE_DTYPE, "err": "missing_image"})
            continue

        dst = out_dir / f"{int(cid)}.npz"
        if dst.exists():
            skip += 1
            rows.append({
                "case_id": int(cid),
                "split": split_name,
                "img_path": pth,
                "npz_path": str(dst),
                "htok": HTOK,
                "wtok": WTOK,
                "dtype": SAVE_DTYPE,
            })
        else:
            try:
                x = load_and_preprocess(pth)
            except Exception as e:
                rows.append({"case_id": int(cid), "split": split_name, "img_path": pth, "npz_path": None,
                             "htok": HTOK, "wtok": WTOK, "dtype": SAVE_DTYPE, "err": str(e)[:200]})
                continue
            buf_case.append(int(cid))
            buf_path.append(pth)
            buf_x.append(x)
            if len(buf_x) >= BATCH:
                flush()

        if idx % 500 == 0:
            elapsed = time.time() - t0
            print(f"[{split_name}] {idx}/{len(case_ids)} | done={done} skip={skip} miss={miss} | {elapsed:.1f}s")

    flush()
    elapsed = time.time() - t0
    print(f"[{split_name}] finished | done={done} skip={skip} miss={miss} | {elapsed:.1f}s")

    return pd.DataFrame(rows)

# ----------------------------
# Run caching
# ----------------------------
tok_train = run_cache(df_train, TRAIN_OUT, "train")
tok_test  = run_cache(df_test,  TEST_OUT,  "test")

tok_train_path = CACHE_ROOT / "tokens_manifest_train.parquet"
tok_test_path  = CACHE_ROOT / "tokens_manifest_test.parquet"
tok_train.to_parquet(tok_train_path, index=False)
tok_test.to_parquet(tok_test_path, index=False)

# globals for later stages
TOKEN_CACHE_ROOT = CACHE_ROOT
TOKEN_MANIFEST_TRAIN = tok_train_path
TOKEN_MANIFEST_TEST  = tok_test_path

print("SAVED:")
print(" -", tok_train_path)
print(" -", tok_test_path)
print("Globals:")
print(" - TOKEN_CACHE_ROOT =", TOKEN_CACHE_ROOT)
print(" - TOKEN_MANIFEST_TRAIN =", TOKEN_MANIFEST_TRAIN)
print(" - TOKEN_MANIFEST_TEST  =", TOKEN_MANIFEST_TEST)


2026-01-11 19:47:15.359434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768160835.606754      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768160835.677584      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768160836.282942      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768160836.282999      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768160836.283002      55 computation_placer.cc:177] computation placer alr

CACHE_ROOT: /kaggle/working/recodai_luc/cache/dinov2_base_518_cfg_292aff008b4d
Train cases: 2795 | Test cases: 1
[train] 500/2795 | done=496 skip=0 miss=0 | 1511.6s


KeyboardInterrupt: 

# Robust Matching (Top-k + MNN + Multi-Peak Translation)

In [ ]:
# ============================================================
# STAGE — Robust Matching (Top-k + MNN + Multi-Peak Translation) (ONE CELL)
# Input: DINOv2 token-grid cache (.npz with key 'tok' -> (Htok,Wtok,D))
# Output (per case_id):
#   /kaggle/working/recodai_luc/cache/match_cfg_<hash>/{train,test}/{case_id}.npz
#     - peaks_dxy : (P,2) int16  [dx,dy]  (token space)
#     - peak_score: (P,)  int32  (#inlier pairs)
#     - src_masks : (P,Htok,Wtok) uint8
#     - tgt_masks : (P,Htok,Wtok) uint8
# Also saves:
#   match_manifest_train.parquet / match_manifest_test.parquet
#
# Notes:
# - CPU-friendly via SimHash grouping + MNN; avoids O(N^2) full KNN.
# - Train split: default only y==1 (forged) to save time.
# ============================================================

import os, json, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Config
# ----------------------------
ONLY_FORGED_TRAIN = True  # recommended (fast)
RUN_TEST = True

SIMHASH_BITS = 12         # fewer bits => larger groups (more recall)
PROJ_DIM = 64             # projection dim for similarity
SEED = 123

SIM_THR = 0.55            # similarity threshold to accept directed NN
MIN_SHIFT = 2             # ignore very small displacement in token space (remove local self-sim)
PEAKS_TOP = 5             # number of translation peaks to keep
PEAK_INLIER_R = 1         # inlier radius around peak (Chebyshev)
NMS_R = 2                 # NMS radius between peaks (Chebyshev)

PROF_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_BASE = Path("/kaggle/working/recodai_luc/cache")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load manifests + find TOKEN cache root
# ----------------------------
paths_json = PROF_DIR / "paths.json"
train_mani = PROF_DIR / "train_manifest.parquet"
test_mani  = PROF_DIR / "test_manifest.parquet"
for p in [paths_json, train_mani, test_mani]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run previous stages first.")

df_train = pd.read_parquet(train_mani)
df_test  = pd.read_parquet(test_mani)

def pick_token_cache_root():
    # 1) globals
    if "TOKEN_CACHE_ROOT" in globals():
        r = Path(str(globals()["TOKEN_CACHE_ROOT"]))
        if r.exists():
            return r
    # 2) auto-search
    cands = sorted(OUT_BASE.glob("dinov2_base_518_cfg_*"))
    cands = [c for c in cands if (c/"cfg.json").exists() and (c/"tokens_manifest_train.parquet").exists()]
    if not cands:
        raise FileNotFoundError("Cannot find DINOv2 token cache under /kaggle/working/recodai_luc/cache. Run DINOv2 Feature Cache stage first.")
    # pick newest modified cfg.json
    cands = sorted(cands, key=lambda p: (p/"cfg.json").stat().st_mtime, reverse=True)
    return cands[0]

TOKEN_ROOT = pick_token_cache_root()
tok_train_pq = TOKEN_ROOT / "tokens_manifest_train.parquet"
tok_test_pq  = TOKEN_ROOT / "tokens_manifest_test.parquet"
if not tok_train_pq.exists():
    raise FileNotFoundError(f"Missing {tok_train_pq}. Run DINOv2 Feature Cache stage first.")

df_tok_train = pd.read_parquet(tok_train_pq)
df_tok_test  = pd.read_parquet(tok_test_pq) if tok_test_pq.exists() else pd.DataFrame()

# Merge y into token train manifest
df_tok_train = df_tok_train.merge(df_train[["case_id","y"]], on="case_id", how="left")
if ONLY_FORGED_TRAIN:
    df_tok_train = df_tok_train[df_tok_train["y"].fillna(0).astype(int).eq(1)].reset_index(drop=True)

# Basic token grid dims (expect consistent)
htok = int(df_tok_train["htok"].dropna().iloc[0])
wtok = int(df_tok_train["wtok"].dropna().iloc[0])
print("TOKEN_ROOT:", TOKEN_ROOT)
print("Token grid:", (htok, wtok))
print("Train tokens:", len(df_tok_train), "| Test tokens:", len(df_tok_test) if RUN_TEST else 0)

# ----------------------------
# Matching cache dirs
# ----------------------------
CFG = {
    "token_root": str(TOKEN_ROOT),
    "simhash_bits": SIMHASH_BITS,
    "proj_dim": PROJ_DIM,
    "seed": SEED,
    "sim_thr": SIM_THR,
    "min_shift": MIN_SHIFT,
    "peaks_top": PEAKS_TOP,
    "peak_inlier_r": PEAK_INLIER_R,
    "nms_r": NMS_R,
    "only_forged_train": ONLY_FORGED_TRAIN,
}
cfg_id = hashlib.sha1(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:12]
MATCH_ROOT = OUT_BASE / f"match_cfg_{cfg_id}"
(MATCH_ROOT / "cfg.json").write_text(json.dumps(CFG, indent=2))
TRAIN_OUT = MATCH_ROOT / "train"
TEST_OUT  = MATCH_ROOT / "test"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)

print("MATCH_ROOT:", MATCH_ROOT)

# ----------------------------
# Core functions
# ----------------------------
def l2norm(x, eps=1e-8):
    n = np.sqrt((x*x).sum(axis=1, keepdims=True)) + eps
    return x / n

def bitpack_signhash(bits_bool: np.ndarray) -> np.ndarray:
    # bits_bool: (N,B) -> uint32 signature
    B = bits_bool.shape[1]
    sig = np.zeros(bits_bool.shape[0], dtype=np.uint32)
    for b in range(B):
        sig |= (bits_bool[:, b].astype(np.uint32) << np.uint32(b))
    return sig

def nms_peaks(hist: np.ndarray, topk: int, r: int):
    # hist: (H,W) int32
    H, W = hist.shape
    h = hist.copy()
    peaks = []
    scores = []
    for _ in range(topk):
        idx = int(np.argmax(h))
        sc = int(h.flat[idx])
        if sc <= 0:
            break
        y, x = divmod(idx, W)
        peaks.append((x, y))
        scores.append(sc)
        y0 = max(0, y - r); y1 = min(H, y + r + 1)
        x0 = max(0, x - r); x1 = min(W, x + r + 1)
        h[y0:y1, x0:x1] = 0
    return peaks, scores

def build_masks_from_pairs(pairs_i, pairs_j, peak_dx, peak_dy, H, W, r_inlier):
    # select inliers around (dx,dy)
    yi, xi = pairs_i // W, pairs_i % W
    yj, xj = pairs_j // W, pairs_j % W
    dx = (xj - xi)
    dy = (yj - yi)
    inl = (np.abs(dx - peak_dx) <= r_inlier) & (np.abs(dy - peak_dy) <= r_inlier)
    if not np.any(inl):
        return None, None, 0
    si = pairs_i[inl]
    sj = pairs_j[inl]
    src = np.zeros((H*W,), dtype=np.uint8)
    tgt = np.zeros((H*W,), dtype=np.uint8)
    src[si] = 1
    tgt[sj] = 1
    return src.reshape(H, W), tgt.reshape(H, W), int(inl.sum())

def robust_match_one(tok_hw_d: np.ndarray):
    # tok_hw_d: (H,W,D) float16/float32
    H, W, D = tok_hw_d.shape
    N = H * W
    X = tok_hw_d.reshape(N, D).astype(np.float32)

    # normalize
    X = l2norm(X)

    # fixed random projections
    rng = np.random.RandomState(SEED)
    R_bits = rng.randn(D, SIMHASH_BITS).astype(np.float32)
    R_proj = rng.randn(D, PROJ_DIM).astype(np.float32)

    bits = (X @ R_bits) > 0
    sig = bitpack_signhash(bits)

    Xp = X @ R_proj
    Xp = l2norm(Xp)

    # group by signature
    order = np.argsort(sig)
    sig_s = sig[order]
    # boundaries
    changes = np.nonzero(sig_s[1:] != sig_s[:-1])[0] + 1
    bounds = np.concatenate(([0], changes, [N]))

    best_j = np.full(N, -1, dtype=np.int32)
    best_s = np.full(N, -1e9, dtype=np.float32)

    for a, b in zip(bounds[:-1], bounds[1:]):
        idx = order[a:b]
        g = len(idx)
        if g < 2:
            continue
        # pairwise sim in projected space within group
        G = Xp[idx]              # (g,PROJ_DIM)
        S = G @ G.T              # (g,g)
        np.fill_diagonal(S, -1e9)
        j_local = np.argmax(S, axis=1)
        s_local = S[np.arange(g), j_local]
        # update
        ii = idx
        jj = idx[j_local]
        upd = s_local > best_s[ii]
        best_s[ii[upd]] = s_local[upd]
        best_j[ii[upd]] = jj[upd]

    # apply similarity threshold
    ok = best_s >= SIM_THR
    ii = np.where(ok & (best_j >= 0))[0]
    jj = best_j[ii]

    # mutual nearest neighbor
    mnn = best_j[jj] == ii
    ii = ii[mnn]
    jj = jj[mnn]
    if len(ii) == 0:
        return {
            "peaks_dxy": np.zeros((0,2), dtype=np.int16),
            "peak_score": np.zeros((0,), dtype=np.int32),
            "src_masks": np.zeros((0,H,W), dtype=np.uint8),
            "tgt_masks": np.zeros((0,H,W), dtype=np.uint8),
        }

    # filter trivial local shifts
    yi, xi = ii // W, ii % W
    yj, xj = jj // W, jj % W
    dx = (xj - xi).astype(np.int32)
    dy = (yj - yi).astype(np.int32)
    not_small = (np.abs(dx) >= MIN_SHIFT) | (np.abs(dy) >= MIN_SHIFT)
    ii = ii[not_small]; jj = jj[not_small]
    if len(ii) == 0:
        return {
            "peaks_dxy": np.zeros((0,2), dtype=np.int16),
            "peak_score": np.zeros((0,), dtype=np.int32),
            "src_masks": np.zeros((0,H,W), dtype=np.uint8),
            "tgt_masks": np.zeros((0,H,W), dtype=np.uint8),
        }

    # displacement histogram in range [-W+1..W-1], [-H+1..H-1]
    dx = (jj % W) - (ii % W)
    dy = (jj // W) - (ii // W)
    dx = dx.astype(np.int32); dy = dy.astype(np.int32)

    dx_min, dx_max = -(W-1), (W-1)
    dy_min, dy_max = -(H-1), (H-1)
    hist = np.zeros((dy_max - dy_min + 1, dx_max - dx_min + 1), dtype=np.int32)
    hx = dx - dx_min
    hy = dy - dy_min
    # safe (should already be in-range)
    valid = (hx >= 0) & (hx < hist.shape[1]) & (hy >= 0) & (hy < hist.shape[0])
    hx = hx[valid]; hy = hy[valid]
    ii2 = ii[valid]; jj2 = jj[valid]
    for x, y in zip(hx, hy):
        hist[y, x] += 1

    # pick peaks with NMS
    (pxy, pscore) = nms_peaks(hist, topk=PEAKS_TOP, r=NMS_R)

    peaks_dxy = []
    peak_score = []
    src_masks = []
    tgt_masks = []

    for (px, py), sc in zip(pxy, pscore):
        peak_dx = int(px + dx_min)
        peak_dy = int(py + dy_min)
        src, tgt, ninl = build_masks_from_pairs(ii2, jj2, peak_dx, peak_dy, H, W, PEAK_INLIER_R)
        if ninl <= 0:
            continue
        peaks_dxy.append([peak_dx, peak_dy])
        peak_score.append(ninl)
        src_masks.append(src)
        tgt_masks.append(tgt)

    if len(peaks_dxy) == 0:
        return {
            "peaks_dxy": np.zeros((0,2), dtype=np.int16),
            "peak_score": np.zeros((0,), dtype=np.int32),
            "src_masks": np.zeros((0,H,W), dtype=np.uint8),
            "tgt_masks": np.zeros((0,H,W), dtype=np.uint8),
        }

    return {
        "peaks_dxy": np.asarray(peaks_dxy, dtype=np.int16),
        "peak_score": np.asarray(peak_score, dtype=np.int32),
        "src_masks": np.asarray(src_masks, dtype=np.uint8),
        "tgt_masks": np.asarray(tgt_masks, dtype=np.uint8),
    }

def load_tok(npz_path: str):
    z = np.load(npz_path)
    if "tok" not in z:
        raise KeyError("npz missing key 'tok'")
    return z["tok"]

def save_match_npz(dst: Path, pack: dict):
    dst.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(dst,
        peaks_dxy=pack["peaks_dxy"],
        peak_score=pack["peak_score"],
        src_masks=pack["src_masks"],
        tgt_masks=pack["tgt_masks"],
    )

def run_split(df_tok: pd.DataFrame, out_dir: Path, split: str):
    rows = []
    t0 = time.time()
    done = skip = fail = 0

    for k, r in df_tok.iterrows():
        cid = int(r["case_id"])
        npz_in = r["npz_path"]
        if not isinstance(npz_in, str) or not Path(npz_in).exists():
            rows.append({"case_id": cid, "split": split, "match_npz": None, "n_peaks": 0, "best_score": 0, "err": "missing_tok"})
            fail += 1
            continue

        dst = out_dir / f"{cid}.npz"
        if dst.exists():
            z = np.load(dst)
            n_peaks = int(z["peaks_dxy"].shape[0]) if "peaks_dxy" in z else 0
            best_score = int(z["peak_score"].max()) if ("peak_score" in z and len(z["peak_score"])>0) else 0
            rows.append({"case_id": cid, "split": split, "match_npz": str(dst), "n_peaks": n_peaks, "best_score": best_score})
            skip += 1
        else:
            try:
                tok = load_tok(npz_in)
                pack = robust_match_one(tok)
                save_match_npz(dst, pack)
                n_peaks = int(pack["peaks_dxy"].shape[0])
                best_score = int(pack["peak_score"].max()) if n_peaks > 0 else 0
                rows.append({"case_id": cid, "split": split, "match_npz": str(dst), "n_peaks": n_peaks, "best_score": best_score})
                done += 1
            except Exception as e:
                rows.append({"case_id": cid, "split": split, "match_npz": None, "n_peaks": 0, "best_score": 0, "err": str(e)[:200]})
                fail += 1

        if (k + 1) % 200 == 0:
            print(f"[{split}] {k+1}/{len(df_tok)} | done={done} skip={skip} fail={fail} | {time.time()-t0:.1f}s")

    print(f"[{split}] finished | done={done} skip={skip} fail={fail} | {time.time()-t0:.1f}s")
    return pd.DataFrame(rows)

# ----------------------------
# Run
# ----------------------------
match_train = run_split(df_tok_train.reset_index(drop=True), TRAIN_OUT, "train")

match_test = pd.DataFrame()
if RUN_TEST and len(df_tok_test):
    match_test = run_split(df_tok_test.reset_index(drop=True), TEST_OUT, "test")

# save manifests
mtrain_pq = MATCH_ROOT / "match_manifest_train.parquet"
match_train.to_parquet(mtrain_pq, index=False)
print("SAVED:", mtrain_pq)

mtest_pq = None
if RUN_TEST and len(match_test):
    mtest_pq = MATCH_ROOT / "match_manifest_test.parquet"
    match_test.to_parquet(mtest_pq, index=False)
    print("SAVED:", mtest_pq)

# globals
MATCH_CACHE_ROOT = MATCH_ROOT
MATCH_MANIFEST_TRAIN = mtrain_pq
MATCH_MANIFEST_TEST = mtest_pq

print("Globals:")
print(" - MATCH_CACHE_ROOT =", MATCH_CACHE_ROOT)
print(" - MATCH_MANIFEST_TRAIN =", MATCH_MANIFEST_TRAIN)
print(" - MATCH_MANIFEST_TEST  =", MATCH_MANIFEST_TEST)


# Verification, Mask Reconstruction & Postprocess

In [ ]:
# ============================================================
# STAGE — Verification, Mask Reconstruction & Postprocess (ONE CELL) — REVISI FULL v8.0
# Fuse:
#   (A) Robust Matching proposals (token-space src/tgt masks)
#   (B) Optional mask-model probability map (token-space or 518-space) if available
# Produce:
#   - per case: final FULL-RES union mask (original HxW) + scalars
#   - per split: pred_features_{train,test}.csv  (for Gate later)
#
# Robustness goals (revisi):
# - KEEP ALL train/test IDs (even if no match/prob => empty mask saved)
# - Safe handling if match_manifest has duplicate case_id (pick best/most recent)
# - Maskprob auto-pick is broader + safer (won't wrongly pick random dirs)
# - Prob alignment robust (accept 37x37, 518x518, any 2D -> resize to token grid)
# - Instance split + filtering in token-space BEFORE upsample (submission-like)
# - Save tok_union for debug; full-res union is the "mask" key
# ============================================================

import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ----------------------------
# Config
# ----------------------------
PROF_DIR = Path("/kaggle/working/recodai_luc_prof")
OUT_BASE = Path("/kaggle/working/recodai_luc/cache")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# Fusion thresholds (token-space probability)
T1 = 0.55          # confident prob
T0 = 0.35          # weak prob (only allowed near seeds)
SEED_DILATE_IT = 1 # token-space dilation iterations

# Component filtering (token-space)
MIN_TOK_AREA = 2
MAX_TOK_AREA_FRAC = 0.80
MAX_INST_KEEP = 8

# Decide "empty/authentic" guard (token-space)
MIN_PEAK_SCORE_KEEP = 6          # if matching weak, likely authentic
MIN_AREA_FRAC_KEEP = 0.0005      # tiny masks -> drop (token-space area frac)

# Optional: allow "prob-only" masks when no match (default OFF for safety)
PROB_ONLY_ENABLE = False
PROB_ONLY_MIN_AREA_FRAC = 0.003   # token-space
PROB_ONLY_MIN_MEAN_PROB = 0.60

# Save
PRED_DIR = OUT_BASE / "pred_ens"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# SciPy optional (faster morphology / CC)
# ----------------------------
try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# Load manifests
# ----------------------------
train_pq = PROF_DIR / "train_manifest.parquet"
test_pq  = PROF_DIR / "test_manifest.parquet"
paths_json = PROF_DIR / "paths.json"
for p in [train_pq, test_pq, paths_json]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run previous stages first.")

df_train = pd.read_parquet(train_pq).copy()
df_test  = pd.read_parquet(test_pq).copy()
PATHS = json.loads(paths_json.read_text())

# Ensure ints
df_train["case_id"] = df_train["case_id"].astype(int)
df_test["case_id"]  = df_test["case_id"].astype(int)

# ----------------------------
# Auto-pick latest MATCH cache root (+ patch size)
# ----------------------------
def pick_latest_match_root():
    if "MATCH_CACHE_ROOT" in globals():
        r = Path(str(globals()["MATCH_CACHE_ROOT"]))
        if r.exists() and (r / "cfg.json").exists():
            return r
    cands = sorted(OUT_BASE.glob("match_cfg_*"))
    cands = [c for c in cands if (c/"cfg.json").exists() and (c/"match_manifest_train.parquet").exists()]
    if not cands:
        raise FileNotFoundError("Cannot find match_cfg_* under /kaggle/working/recodai_luc/cache. Run Robust Matching stage first.")
    cands = sorted(cands, key=lambda p: (p/"cfg.json").stat().st_mtime, reverse=True)
    return cands[0]

MATCH_ROOT = pick_latest_match_root()
MATCH_CFG = json.loads((MATCH_ROOT / "cfg.json").read_text())
PATCH = int(MATCH_CFG.get("patch", MATCH_CFG.get("patch_size", 14)))
HTOK = int(MATCH_CFG.get("Ht", MATCH_CFG.get("htok", 37)))
WTOK = int(MATCH_CFG.get("Wt", MATCH_CFG.get("wtok", 37)))

mtrain_pq = MATCH_ROOT / "match_manifest_train.parquet"
mtest_pq  = MATCH_ROOT / "match_manifest_test.parquet"
df_mtrain = pd.read_parquet(mtrain_pq)
df_mtest  = pd.read_parquet(mtest_pq) if mtest_pq.exists() else pd.DataFrame(columns=["case_id","match_npz"])

# sanitize manifest cols
for dfm in [df_mtrain, df_mtest]:
    if "case_id" in dfm.columns:
        dfm["case_id"] = dfm["case_id"].astype(int)
    if "match_npz" not in dfm.columns:
        dfm["match_npz"] = None

print("MATCH_ROOT:", MATCH_ROOT)
print("MATCH_CFG :", {"PATCH": PATCH, "HTOK": HTOK, "WTOK": WTOK})
print("mtrain:", len(df_mtrain), "| mtest:", len(df_mtest))

# ----------------------------
# Pick best match_npz per case_id if duplicates exist
# ----------------------------
def pick_best_match_paths(df_match: pd.DataFrame):
    if df_match is None or len(df_match) == 0:
        return {}
    dfm = df_match.copy()
    if "match_npz" not in dfm.columns:
        return {}
    dfm = dfm[dfm["match_npz"].notna()].copy()
    if len(dfm) == 0:
        return {}
    # If manifest already has a score column, use it
    score_cols = [c for c in ["best_peak_score","peak_score_max","max_peak_score","score_max","best_score"] if c in dfm.columns]
    if score_cols:
        sc = score_cols[0]
        dfm[sc] = pd.to_numeric(dfm[sc], errors="coerce").fillna(-1)
        dfm = dfm.sort_values([ "case_id", sc ], ascending=[True, False])
        dfm = dfm.drop_duplicates("case_id", keep="first")
        return dfm.set_index("case_id")["match_npz"].to_dict()

    # Else: pick most recently modified existing file
    def _mtime(p):
        try:
            return Path(p).stat().st_mtime
        except Exception:
            return -1
    dfm["_mtime"] = dfm["match_npz"].map(_mtime)
    dfm = dfm.sort_values(["case_id","_mtime"], ascending=[True, False])
    dfm = dfm.drop_duplicates("case_id", keep="first")
    return dfm.set_index("case_id")["match_npz"].to_dict()

match_map_train = pick_best_match_paths(df_mtrain)
match_map_test  = pick_best_match_paths(df_mtest)

# ----------------------------
# Optional: auto-pick mask-prob dir (contains {case_id}.npz)
# ----------------------------
def auto_pick_maskprob_dir(sample_case_ids):
    # candidate roots (fast + targeted)
    roots = []
    roots.append(OUT_BASE)
    roots.append(OUT_BASE / "dino_v2")
    roots.append(OUT_BASE / "mask_prob")
    roots.append(OUT_BASE / "pred_mask")
    roots.append(OUT_BASE / "pred_tok")
    roots.append(OUT_BASE / "seg_prob")
    roots.append(OUT_BASE / "mask_model")
    roots = [r for r in roots if r.exists() and r.is_dir()]

    # candidates: common patterns in roots + one-level nested
    cands = []
    pats = ["mask_prob_*","pred_mask_*","pred_tok_*","pred_base_*","pred_giant_*","seg_prob_*","maskprob_*","prob_*","maskdl_*"]
    for rt in roots:
        for pat in pats:
            cands += list(rt.glob(pat))
        for d in rt.glob("*"):
            if d.is_dir():
                for pat in pats:
                    cands += list(d.glob(pat))

    # unique dirs only
    uniq = []
    seen = set()
    for d in cands:
        if d.is_dir():
            s = str(d.resolve())
            if s not in seen:
                uniq.append(d); seen.add(s)

    # score by how many sample ids exist as <cid>.npz
    best = None
    best_hit = 0
    for d in uniq:
        hit = 0
        for cid in sample_case_ids[:40]:
            if (d / f"{int(cid)}.npz").exists():
                hit += 1
        if hit > best_hit:
            best_hit = hit
            best = d
    return best, best_hit

sample_ids = df_train["case_id"].head(60).tolist()
MASKPROB_DIR, hit = auto_pick_maskprob_dir(sample_ids)
if MASKPROB_DIR is not None and hit >= 5:
    print("MASKPROB_DIR picked:", MASKPROB_DIR, f"(hits on samples={hit})")
else:
    MASKPROB_DIR = None
    print("MASKPROB_DIR: None (no reliable cache found)")

# ----------------------------
# Utils: morphology / CC in token-space
# ----------------------------
def dilate_tok(x_bool, it=1):
    if it <= 0:
        return x_bool
    x = x_bool.astype(bool)
    if _HAS_SCIPY:
        return ndi.binary_dilation(x, iterations=it)
    # fallback: 3x3 max-pool style
    for _ in range(it):
        xp = np.pad(x, 1, mode="constant", constant_values=False)
        y = np.zeros_like(x, dtype=bool)
        for dy in (-1,0,1):
            for dx in (-1,0,1):
                y |= xp[1+dy:1+dy+x.shape[0], 1+dx:1+dx+x.shape[1]]
        x = y
    return x

def label_cc(x_bool):
    x = x_bool.astype(bool)
    if _HAS_SCIPY:
        lab, n = ndi.label(x, structure=np.ones((3,3), dtype=np.uint8))
        return lab, int(n)
    # fallback BFS
    H, W = x.shape
    lab = np.zeros((H,W), dtype=np.int32)
    cur = 0
    for y in range(H):
        for x0 in range(W):
            if (not x[y,x0]) or lab[y,x0] != 0:
                continue
            cur += 1
            stack = [(y,x0)]
            lab[y,x0] = cur
            while stack:
                yy, xx = stack.pop()
                for dy in (-1,0,1):
                    for dx in (-1,0,1):
                        if dy==0 and dx==0:
                            continue
                        ny, nx = yy+dy, xx+dx
                        if 0 <= ny < H and 0 <= nx < W and x[ny,nx] and lab[ny,nx]==0:
                            lab[ny,nx] = cur
                            stack.append((ny,nx))
    return lab, int(cur)

def upsample_tok_to_518(x_hw, patch=14):
    # nearest by repeat: (Ht,Wt)->(Ht*patch, Wt*patch)
    return np.kron(x_hw.astype(np.uint8), np.ones((patch,patch), dtype=np.uint8))

def resize_to_hw_bool(x_uint8_01, H, W):
    im = Image.fromarray((x_uint8_01.astype(np.uint8) * 255))
    im = im.resize((int(W), int(H)), resample=Image.NEAREST)
    return (np.asarray(im) > 127).astype(np.uint8)

# ----------------------------
# Load match npz (robust keys)
# ----------------------------
def load_match_npz(p):
    z = np.load(p)
    peaks = z["peaks_dxy"] if "peaks_dxy" in z.files else np.zeros((0,2), np.int16)
    scores = z["peak_score"] if "peak_score" in z.files else np.zeros((0,), np.int32)
    src = z["src_masks"] if "src_masks" in z.files else np.zeros((0,HTOK,WTOK), np.uint8)
    tgt = z["tgt_masks"] if "tgt_masks" in z.files else np.zeros((0,HTOK,WTOK), np.uint8)
    return peaks, scores, src, tgt

# ----------------------------
# Load maskprob npz (robust keys)
# ----------------------------
def load_maskprob_npz(case_id: int):
    if MASKPROB_DIR is None:
        return None
    p = MASKPROB_DIR / f"{int(case_id)}.npz"
    if not p.exists():
        return None
    z = np.load(p)
    # try common keys
    for k in ["prob_tok","p_tok","prob","p","mask_prob","pred","logits","probs","mask"]:
        if k in z.files:
            return z[k]
    # fallback: first array
    keys = list(z.files)
    return z[keys[0]] if keys else None

def align_prob_to_tok(prob_any):
    if prob_any is None:
        return None
    a = np.asarray(prob_any)
    # squeeze trivial dims
    while a.ndim > 2 and a.shape[0] == 1:
        a = a[0]
    if a.ndim != 2:
        return None
    # if already token grid
    if a.shape == (HTOK, WTOK):
        return a.astype(np.float32)
    # if 518-like and divisible by PATCH, do block-mean
    if a.shape[0] % PATCH == 0 and a.shape[1] % PATCH == 0:
        h = a.shape[0] // PATCH
        w = a.shape[1] // PATCH
        if (h, w) == (HTOK, WTOK):
            p = a.astype(np.float32).reshape(HTOK, PATCH, WTOK, PATCH).mean(axis=(1,3))
            return p
    # fallback: resize to token grid (bilinear)
    im = Image.fromarray(a.astype(np.float32))
    im = im.resize((WTOK, HTOK), resample=Image.BILINEAR)
    return np.asarray(im).astype(np.float32)

# ----------------------------
# Core: build token instances from (match seeds + optional prob)
# ----------------------------
def build_token_instances(src_masks, tgt_masks, peak_score, prob_any):
    has_match = int(len(peak_score) > 0 and src_masks.shape[0] > 0 and tgt_masks.shape[0] > 0)
    best_score = int(np.max(peak_score)) if has_match else 0

    # seed union
    if has_match:
        seed = (src_masks.astype(bool) | tgt_masks.astype(bool)).any(axis=0)
        seed = seed[:HTOK, :WTOK]
    else:
        seed = np.zeros((HTOK, WTOK), dtype=bool)

    # prob_tok
    prob_tok = align_prob_to_tok(prob_any)
    has_prob = int(prob_tok is not None)
    mean_prob = float(np.mean(prob_tok)) if has_prob else np.nan

    # fusion
    if has_prob:
        hard = (prob_tok >= T1)
        soft = (prob_tok >= T0)
        seed_d = dilate_tok(seed, SEED_DILATE_IT)
        fused = hard | (seed_d & soft)

        # optional prob-only if no match
        if (not has_match) and PROB_ONLY_ENABLE:
            hard_area_frac = float(hard.mean())
            if (hard_area_frac >= PROB_ONLY_MIN_AREA_FRAC) and (mean_prob >= PROB_ONLY_MIN_MEAN_PROB):
                fused = hard.copy()
            else:
                fused = np.zeros((HTOK,WTOK), dtype=bool)
    else:
        fused = seed.copy()

    # connected components -> instances with filters
    lab, ncc = label_cc(fused)
    insts, areas = [], []
    for k in range(1, ncc+1):
        m = (lab == k)
        a = int(m.sum())
        if a < MIN_TOK_AREA:
            continue
        if a / float(HTOK*WTOK) > MAX_TOK_AREA_FRAC:
            continue
        insts.append(m.astype(np.uint8))
        areas.append(a)

    if len(insts) == 0:
        return {
            "mask_tok_inst": np.zeros((0,HTOK,WTOK), dtype=np.uint8),
            "mask_tok_union": np.zeros((HTOK,WTOK), dtype=np.uint8),
            "n_inst": 0,
            "area_frac_tok": 0.0,
            "best_peak_score": best_score,
            "mean_prob_tok": mean_prob,
            "has_match": has_match,
            "has_prob": has_prob,
        }

    # keep top-K by area
    order = np.argsort(np.asarray(areas))[::-1][:MAX_INST_KEEP]
    mask_tok_inst = np.stack([insts[i] for i in order], axis=0).astype(np.uint8)
    union = (mask_tok_inst.any(axis=0)).astype(np.uint8)
    area_frac = float(union.mean())

    # final guard: weak match + tiny area -> drop
    if (best_score < MIN_PEAK_SCORE_KEEP) and (area_frac < MIN_AREA_FRAC_KEEP):
        return {
            "mask_tok_inst": np.zeros((0,HTOK,WTOK), dtype=np.uint8),
            "mask_tok_union": np.zeros((HTOK,WTOK), dtype=np.uint8),
            "n_inst": 0,
            "area_frac_tok": 0.0,
            "best_peak_score": best_score,
            "mean_prob_tok": mean_prob,
            "has_match": has_match,
            "has_prob": has_prob,
        }

    return {
        "mask_tok_inst": mask_tok_inst,
        "mask_tok_union": union,
        "n_inst": int(mask_tok_inst.shape[0]),
        "area_frac_tok": area_frac,
        "best_peak_score": best_score,
        "mean_prob_tok": mean_prob,
        "has_match": has_match,
        "has_prob": has_prob,
    }

# ----------------------------
# Run split (always writes npz for every case_id)
# ----------------------------
def run_split(df_cases: pd.DataFrame, match_map: dict, split: str):
    out_split = PRED_DIR / split
    out_split.mkdir(parents=True, exist_ok=True)

    rows_feat = []
    t0 = time.time()
    done = skip = rebuilt = 0

    for j, r in enumerate(df_cases.itertuples(index=False), start=1):
        cid = int(getattr(r, "case_id"))
        H = int(getattr(r, "H")) if hasattr(r, "H") and pd.notna(getattr(r, "H")) else 518
        W = int(getattr(r, "W")) if hasattr(r, "W") and pd.notna(getattr(r, "W")) else 518

        dst = out_split / f"{cid}.npz"
        if dst.exists():
            # fast read scalars, but ensure minimal keys exist
            try:
                z = np.load(dst)
                if ("mask" in z.files) and ("n_inst" in z.files) and ("best_peak_score" in z.files):
                    rows_feat.append({
                        "case_id": cid,
                        "split": split,
                        "n_inst": int(z["n_inst"]) if "n_inst" in z.files else 0,
                        "area_frac": float(z["area_frac"]) if "area_frac" in z.files else 0.0,
                        "area_frac_tok": float(z["area_frac_tok"]) if "area_frac_tok" in z.files else 0.0,
                        "best_peak_score": int(z["best_peak_score"]) if "best_peak_score" in z.files else 0,
                        "has_match": int(z["has_match"]) if "has_match" in z.files else 0,
                        "has_prob": int(z["has_prob"]) if "has_prob" in z.files else 0,
                        "mean_prob_tok": float(z["mean_prob_tok"]) if "mean_prob_tok" in z.files else np.nan,
                        "match_exists": int(z["match_exists"]) if "match_exists" in z.files else (1 if cid in match_map else 0),
                        "prob_exists": int(z["prob_exists"]) if "prob_exists" in z.files else (1 if (MASKPROB_DIR is not None and (MASKPROB_DIR/f"{cid}.npz").exists()) else 0),
                        "npz_path": str(dst),
                    })
                    skip += 1
                    continue
            except Exception:
                pass  # rebuild

        mp = match_map.get(cid, None)
        match_exists = int(isinstance(mp, str) and Path(mp).exists())
        prob_any = load_maskprob_npz(cid)
        prob_exists = int(prob_any is not None)

        if match_exists:
            _, scores, src, tgt = load_match_npz(mp)
            pack_tok = build_token_instances(src, tgt, scores, prob_any)
        else:
            # no match: still allow prob-only if enabled, else empty
            pack_tok = build_token_instances(
                np.zeros((0,HTOK,WTOK), np.uint8),
                np.zeros((0,HTOK,WTOK), np.uint8),
                np.zeros((0,), np.int32),
                prob_any
            )

        # token union -> 518 -> original HxW
        tok_union = pack_tok["mask_tok_union"]
        mask_518 = upsample_tok_to_518(tok_union, patch=PATCH)  # (HTOK*PATCH, WTOK*PATCH)
        mask_full = resize_to_hw_bool(mask_518, H, W)
        area_frac_full = float(mask_full.mean())

        # save
        np.savez_compressed(
            dst,
            mask=mask_full.astype(np.uint8),        # FULL-RES union (HxW)
            tok_union=tok_union.astype(np.uint8),   # token union (HTOKxWTOK) for debug
            H=int(H), W=int(W),
            n_inst=int(pack_tok["n_inst"]),
            area_frac=float(area_frac_full),
            area_frac_tok=float(pack_tok["area_frac_tok"]),
            best_peak_score=int(pack_tok["best_peak_score"]),
            has_match=int(pack_tok["has_match"]),
            has_prob=int(pack_tok["has_prob"]),
            mean_prob_tok=float(pack_tok["mean_prob_tok"]) if np.isfinite(pack_tok["mean_prob_tok"]) else np.nan,
            match_exists=int(match_exists),
            prob_exists=int(prob_exists),
        )

        rows_feat.append({
            "case_id": cid,
            "split": split,
            "n_inst": int(pack_tok["n_inst"]),
            "area_frac": float(area_frac_full),
            "area_frac_tok": float(pack_tok["area_frac_tok"]),
            "best_peak_score": int(pack_tok["best_peak_score"]),
            "has_match": int(pack_tok["has_match"]),
            "has_prob": int(pack_tok["has_prob"]),
            "mean_prob_tok": float(pack_tok["mean_prob_tok"]) if np.isfinite(pack_tok["mean_prob_tok"]) else np.nan,
            "match_exists": int(match_exists),
            "prob_exists": int(prob_exists),
            "npz_path": str(dst),
        })
        rebuilt += 1

        if j % 300 == 0:
            print(f"[{split}] {j}/{len(df_cases)} | rebuilt={rebuilt} skip={skip} | {time.time()-t0:.1f}s")

    print(f"[{split}] finished | rebuilt={rebuilt} skip={skip} | {time.time()-t0:.1f}s")
    return pd.DataFrame(rows_feat)

# ----------------------------
# Run train/test
# ----------------------------
df_cases_train = df_train[["case_id","H","W","y"]].copy()
df_cases_test  = df_test[["case_id","H","W"]].copy()

feat_train = run_split(df_cases_train, match_map_train, "train")
feat_test  = run_split(df_cases_test,  match_map_test,  "test")

# Save features (keep under PRED_DIR for Gate stages)
feat_train_path = PRED_DIR / "pred_features_train.csv"
feat_test_path  = PRED_DIR / "pred_features_test.csv"
feat_train.to_csv(feat_train_path, index=False)
feat_test.to_csv(feat_test_path, index=False)

print("SAVED:")
print(" -", feat_train_path)
print(" -", feat_test_path)
print("PRED_DIR:", PRED_DIR)

# Globals
PRED_ENS_DIR = PRED_DIR
PRED_FEATURES_TRAIN = feat_train_path
PRED_FEATURES_TEST  = feat_test_path
MATCH_CACHE_ROOT = MATCH_ROOT
MASKPROB_DIR_USED = MASKPROB_DIR
